# M6/T1 Homework: vLLM, guided decoding и параметры генерации

В этой работе я поднимаю локальный OpenAI-compatible сервер через `vllm serve`,
отправляю обычный chat-запрос, затем проверяю structured outputs:
JSON Schema, choice, regex и grammar. В конце сравниваю ответы при разных
параметрах генерации.

## 1. План и зависимости

Задание из `task.md`:
- поднять модель через vLLM и отправить запрос;
- сделать guided JSON запрос;
- сделать другие guided запросы;
- сравнить генерации при разных `top_p`, `top_k`, `temperature`, `max_tokens`.

Зависимости добавлены через `uv add` в корневой `pyproject.toml`:
`openai`, `pydantic`, `requests`, `vllm`.

Для 8 GB VRAM выбран компактный `Qwen/Qwen3-0.6B`.

In [1]:
import atexit
import os
import random
import re
import socket
import subprocess
import time
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from openai import OpenAI
from pydantic import BaseModel, ConfigDict, Field

SEED = 42
random.seed(SEED)

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "T1" else Path("M6/T1").resolve()
HF_HOME = NOTEBOOK_DIR / ".cache" / "huggingface"
VLLM_LOG = NOTEBOOK_DIR / "vllm_server.log"

MODEL_ID = os.environ.get("VLLM_MODEL", "Qwen/Qwen3-0.6B")
HOST = os.environ.get("VLLM_HOST", "127.0.0.1")
PORT = int(os.environ.get("VLLM_PORT", "8000"))
BASE_URL = f"http://{HOST}:{PORT}/v1"

SERVER_PROCESS: subprocess.Popen[str] | None = None

print("Model:", MODEL_ID)
print("Base URL:", BASE_URL)
print("HF_HOME:", HF_HOME)


def check(condition: bool, success_message: str, failure_message: str | None = None) -> None:
    """Print an explicit check result and stop the notebook on failure."""
    if condition:
        print(f"[OK] {success_message}")
        return
    raise RuntimeError(f"[FAIL] {failure_message or success_message}")

Model: Qwen/Qwen3-0.6B
Base URL: http://127.0.0.1:8000/v1
HF_HOME: /home/krv/repo/workspace/LLM_course/M6/T1/.cache/huggingface


## 2. Запуск vLLM-сервера

Этот блок идемпотентный:
если сервер уже поднят на `BASE_URL`, новый процесс не стартует.
Иначе запускается команда вида:

```bash
uv run vllm serve Qwen/Qwen3-0.6B \
  --host 127.0.0.1 \
  --port 8000 \
  --gpu-memory-utilization 0.82 \
  --max-model-len 2048 \
  --trust-remote-code
```

In [2]:
def port_is_open(host: str, port: int, timeout: float = 1.0) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        return sock.connect_ex((host, port)) == 0


def server_is_ready(base_url: str = BASE_URL) -> bool:
    try:
        response = requests.get(f"{base_url}/models", timeout=2)
        return response.ok and bool(response.json().get("data"))
    except requests.RequestException:
        return False


def start_vllm_server() -> subprocess.Popen[str]:
    HF_HOME.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env["HF_HOME"] = str(HF_HOME)
    env.setdefault("CUDA_VISIBLE_DEVICES", "0")

    command = [
        "uv",
        "run",
        "vllm",
        "serve",
        MODEL_ID,
        "--host",
        HOST,
        "--port",
        str(PORT),
        "--gpu-memory-utilization",
        "0.82",
        "--max-model-len",
        "2048",
        "--trust-remote-code",
    ]

    log_file = VLLM_LOG.open("w", encoding="utf-8")
    process = subprocess.Popen(
        command,
        cwd=Path.cwd(),
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print("Started:", " ".join(command))
    print("Log:", VLLM_LOG)
    return process


def wait_for_vllm(process: subprocess.Popen[str] | None, timeout_s: int = 600) -> None:
    started_at = time.time()
    while time.time() - started_at < timeout_s:
        if server_is_ready():
            print("vLLM server is ready.")
            return
        if process is not None and process.poll() is not None:
            tail = VLLM_LOG.read_text(encoding="utf-8")[-4000:] if VLLM_LOG.exists() else ""
            raise RuntimeError(f"vLLM process exited with code {process.returncode}.\n\nLog tail:\n{tail}")
        time.sleep(5)

    tail = VLLM_LOG.read_text(encoding="utf-8")[-4000:] if VLLM_LOG.exists() else ""
    raise TimeoutError(f"vLLM server did not become ready in {timeout_s}s.\n\nLog tail:\n{tail}")


def stop_vllm_server() -> None:
    global SERVER_PROCESS
    if SERVER_PROCESS is not None and SERVER_PROCESS.poll() is None:
        SERVER_PROCESS.terminate()
        try:
            SERVER_PROCESS.wait(timeout=30)
        except subprocess.TimeoutExpired:
            SERVER_PROCESS.kill()
    SERVER_PROCESS = None


atexit.register(stop_vllm_server)

if server_is_ready():
    print("Using already running vLLM server.")
elif port_is_open(HOST, PORT):
    raise RuntimeError(f"Port {HOST}:{PORT} is open, but /v1/models is not ready.")
else:
    SERVER_PROCESS = start_vllm_server()
    wait_for_vllm(SERVER_PROCESS)

Started: uv run vllm serve Qwen/Qwen3-0.6B --host 127.0.0.1 --port 8000 --gpu-memory-utilization 0.82 --max-model-len 2048 --trust-remote-code
Log: /home/krv/repo/workspace/LLM_course/M6/T1/vllm_server.log
vLLM server is ready.


## 3. Клиент и обычный запрос

In [3]:
client = OpenAI(api_key="EMPTY", base_url=BASE_URL)

models = client.models.list()
served_model = models.data[0].id
print("Served model:", served_model)

Served model: Qwen/Qwen3-0.6B


In [4]:
basic_response = client.chat.completions.create(
    model=served_model,
    messages=[
        {"role": "system", "content": "You answer briefly and concretely."},
        {"role": "user", "content": "In one sentence, explain what guided decoding does in vLLM."},
    ],
    temperature=0.2,
    max_tokens=80,
)

basic_text = basic_response.choices[0].message.content
print(basic_text)

check(
    bool(basic_text and len(basic_text) > 20),
    "Basic request returned a non-empty explanatory answer.",
    "Basic request returned an empty or suspiciously short answer.",
)

<think>
Okay, the user is asking for a one-sentence explanation of what guided decoding does in vLLM. Let me start by recalling what guided decoding is. From what I remember, guided decoding is a technique used in machine learning, particularly in tasks like classification or regression, where the model is trained to make decisions based on certain features. It's often used in deep learning models to improve
[OK] Basic request returned a non-empty explanatory answer.


## 4. Guided JSON

Здесь vLLM получает JSON Schema из Pydantic-модели. После генерации ответ
валидируется локально через `model_validate_json`.

In [5]:
class RoutePlan(BaseModel):
    model_config = ConfigDict(extra="forbid")

    destination: str = Field(description="Target city")
    days: int = Field(ge=1, le=7)
    activities: list[str] = Field(min_length=2, max_length=4)
    budget_level: str = Field(pattern="^(low|medium|high)$")


json_completion = client.chat.completions.create(
    model=served_model,
    messages=[
        {
            "role": "system",
            "content": "Return only a compact JSON object that matches the provided schema.",
        },
        {
            "role": "user",
            "content": "Plan a 3-day learning-focused trip to Berlin for an LLM engineering student.",
        },
    ],
    temperature=0.1,
    max_tokens=180,
    extra_body={"structured_outputs": {"json": RoutePlan.model_json_schema()}},
)

json_text = json_completion.choices[0].message.content
route_plan = RoutePlan.model_validate_json(json_text)
print(json_text)
print(route_plan.model_dump_json(indent=2))

check(
    isinstance(route_plan.days, int) and 1 <= route_plan.days <= 7,
    "Guided JSON contains a valid days field.",
    "Guided JSON has an invalid days field.",
)
check(
    len(route_plan.activities) >= 2,
    "Guided JSON contains at least two planned activities.",
    "Guided JSON does not contain enough activities.",
)

{"destination": "Berlin", "days": 3, "activities": ["Day 1: Explore the Berlin Wall and the Berlin Wall Museum", "Day 2: Visit the Reichstag and the Berlin Wall Museum", "Day 3: Attend a lecture on AI and Machine Learning at the University of Berlin"], "budget_level": "medium"}
{
  "destination": "Berlin",
  "days": 3,
  "activities": [
    "Day 1: Explore the Berlin Wall and the Berlin Wall Museum",
    "Day 2: Visit the Reichstag and the Berlin Wall Museum",
    "Day 3: Attend a lecture on AI and Machine Learning at the University of Berlin"
  ],
  "budget_level": "medium"
}
[OK] Guided JSON contains a valid days field.
[OK] Guided JSON contains at least two planned activities.


## 5. Другие guided-запросы

Проверяю три разных ограничения:
- `choice`: ответ только из фиксированного списка;
- `regex`: ответ должен соответствовать регулярному выражению;
- `grammar`: генерация SQL из маленькой EBNF-грамматики.

In [6]:
choice_completion = client.chat.completions.create(
    model=served_model,
    messages=[
        {"role": "user", "content": "Classify sentiment: vLLM structured outputs are useful."},
    ],
    temperature=0.0,
    max_tokens=8,
    extra_body={"structured_outputs": {"choice": ["positive", "negative", "neutral"]}},
)

choice_text = choice_completion.choices[0].message.content
print("choice:", choice_text)

check(
    choice_text in {"positive", "negative", "neutral"},
    "Guided choice returned one of the allowed labels.",
    f"Guided choice returned an unexpected label: {choice_text!r}.",
)

choice: neutral
[OK] Guided choice returned one of the allowed labels.


In [7]:
regex_completion = client.chat.completions.create(
    model=served_model,
    messages=[
        {"role": "user", "content": "Generate one warehouse SKU for a transformer checkpoint."},
    ],
    temperature=0.0,
    max_tokens=12,
    extra_body={"structured_outputs": {"regex": r"SKU-[0-9]{3}"}},
)

regex_text = regex_completion.choices[0].message.content
print("regex:", regex_text)

check(
    re.fullmatch(r"SKU-\d{3}", regex_text or "") is not None,
    "Guided regex returned a SKU in the required format.",
    f"Guided regex returned an invalid SKU: {regex_text!r}.",
)

regex: SKU-100
[OK] Guided regex returned a SKU in the required format.


In [8]:
simple_sql_grammar = """
root ::= select_statement
select_statement ::= "SELECT " column " FROM " table " WHERE " condition
column ::= "model_name" | "latency_ms" | "quality_score"
table ::= "runs" | "models"
condition ::= column " > " number
number ::= "10" | "50" | "90"
"""

grammar_completion = client.chat.completions.create(
    model=served_model,
    messages=[
        {"role": "user", "content": "Generate a SQL query for high quality model runs."},
    ],
    temperature=0.0,
    max_tokens=60,
    extra_body={"structured_outputs": {"grammar": simple_sql_grammar}},
)

grammar_text = grammar_completion.choices[0].message.content
print("grammar:", grammar_text)

check(
    grammar_text.startswith("SELECT ") and " FROM " in grammar_text and " WHERE " in grammar_text,
    "Guided grammar returned a SQL query matching the expected shape.",
    f"Guided grammar returned an invalid SQL query: {grammar_text!r}.",
)

grammar: SELECT quality_score FROM models WHERE quality_score > 90
[OK] Guided grammar returned a SQL query matching the expected shape.


## 6. Разные параметры генерации

Сравниваю четыре режима. `top_k` передаётся через `extra_body`, потому что
это vLLM-specific параметр OpenAI-compatible endpoint.

In [9]:
generation_prompt = (
    "Give a concise practical tip for choosing generation parameters when serving a small LLM with vLLM."
)

generation_configs: list[dict[str, Any]] = [
    {
        "name": "deterministic",
        "temperature": 0.0,
        "top_p": 1.0,
        "top_k": 1,
        "max_tokens": 70,
    },
    {
        "name": "balanced",
        "temperature": 0.5,
        "top_p": 0.9,
        "top_k": 40,
        "max_tokens": 90,
    },
    {
        "name": "diverse_short",
        "temperature": 0.9,
        "top_p": 0.95,
        "top_k": 80,
        "max_tokens": 55,
    },
    {
        "name": "diverse_long",
        "temperature": 1.0,
        "top_p": 0.98,
        "top_k": 100,
        "max_tokens": 130,
    },
]

rows = []
for config in generation_configs:
    completion = client.chat.completions.create(
        model=served_model,
        messages=[{"role": "user", "content": generation_prompt}],
        temperature=config["temperature"],
        top_p=config["top_p"],
        max_tokens=config["max_tokens"],
        extra_body={"top_k": config["top_k"]},
    )
    text = completion.choices[0].message.content or ""
    rows.append(
        {
            **config,
            "chars": len(text),
            "answer": text,
        }
    )

generation_results = pd.DataFrame(rows)
print(generation_results.to_string(index=False))

check(
    generation_results.shape[0] == len(generation_configs),
    "Generation comparison produced one row per parameter set.",
    "Generation comparison row count does not match the number of parameter sets.",
)
check(
    generation_results["chars"].gt(0).all(),
    "Every generation parameter set produced a non-empty answer.",
    "At least one generation parameter set produced an empty answer.",
)

print("All M6/T1 checks passed.")

         name  temperature  top_p  top_k  max_tokens  chars                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                answer
deterministic          0.0   1.00      1          70    316                                                                                                                                                                                                                                           

## 7. Выводы
- Сервер vLLM успешно поднялся с моделью `Qwen/Qwen3-0.6B`, notebook получил
  модель через `/v1/models` и выполнил все запросы через OpenAI-compatible API.
- Для обычных запросов Qwen3 возвращал текст с `<think>`-секцией. Это полезное
  практическое наблюдение: если нужен только финальный ответ без reasoning text,
  сервер стоит запускать с reasoning parser или дополнительно чистить вывод.
- Guided JSON сработал корректно: модель вернула валидный объект для поездки
  в Berlin на 3 дня с `budget_level="medium"`, и Pydantic успешно провалидировал
  результат.
- Остальные structured outputs также соблюли ограничения: `choice` вернул
  `neutral`, `regex` вернул `SKU-100`, а grammar-запрос вернул SQL вида
  `SELECT quality_score FROM models WHERE quality_score > 90`.
- В сравнении параметров все четыре режима дали непустые ответы. Более высокий
  `max_tokens` ожидаемо увеличил длину вывода: короткий diverse-режим дал 252
  символа, а `diverse_long` с `max_tokens=130` дал 642 символа.
- Главный вывод по guided decoding: даже когда свободная генерация содержит
  служебный reasoning text, structured outputs остаются удобным способом
  получить контролируемый формат для downstream-кода.

## 8. Остановка сервера

Если сервер был запущен этим ноутбуком, следующая клетка освободит GPU.
Если сервер был запущен заранее, она ничего не остановит.

In [10]:
stop_vllm_server()
print("vLLM server process owned by this notebook is stopped.")

vLLM server process owned by this notebook is stopped.
